# Notebook 03 — Training Tricks Ablation

Paper's Fig. 3(a): each training trick (teacher forcing, Markov, improved
residuals, noise + normalization + cosine decay = "bag of tricks") adds on
top of the previous and pushes N-MSE down. We replicate a scaled-down
version of this study.

**What we can vary in our pipeline:**
- Input Gaussian noise (on/off)
- Cosine LR decay (on/off)
- Residual connections (FNO = off, F-FNO = on, can't toggle without new code)
- Factorized spectral layer (this is the architectural difference, use F-FNO vs FNO)

**What is always on here:** Markov assumption and teacher forcing (both
are built into our `NavierStokesDataset`, which samples `(w_t, w_{t+1})`
pairs and feeds ground-truth `w_t` during training).

This matches the paper structure — `FNO-TF` and `FNO-M` in their Fig. 3
are baselines we already have in both models.

We fix F-FNO architecture (8 layers) and toggle the tricks one at a time.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from train import train, TrainConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = PROJECT_ROOT / 'ns_data'
RUNS_DIR = PROJECT_ROOT / 'runs' / 'ablation'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

## Ablation configurations

Start from a baseline with all tricks **off**, then add them one by one,
ending at the full F-FNO++ setup.

In [ ]:
# Ablation grid. Each config differs by one flag from the previous row.
ABLATION_CONFIGS = [
    {'name': 'FNO (plain)',       'model_type': 'fno',  'noise': 0.0,  'cosine': False},
    {'name': 'FNO + cosine LR',   'model_type': 'fno',  'noise': 0.0,  'cosine': True},
    {'name': 'FNO + cosine + noise', 'model_type': 'fno', 'noise': 1e-3, 'cosine': True},
    {'name': 'F-FNO (plain)',     'model_type': 'ffno', 'noise': 0.0,  'cosine': False},
    {'name': 'F-FNO + cosine LR', 'model_type': 'ffno', 'noise': 0.0,  'cosine': True},
    {'name': 'F-FNO + cosine + noise', 'model_type': 'ffno', 'noise': 1e-3, 'cosine': True},
]

# Shared settings — fix everything else
N_LAYERS = 8
N_EPOCHS = 15
BATCH_SIZE = 20
HIDDEN = 32
MODES = 12

In [ ]:
ablation_results = {}
for i, cfg_dict in enumerate(ABLATION_CONFIGS):
    name = cfg_dict['name']
    print(f'\n{"="*60}\n[{i+1}/{len(ABLATION_CONFIGS)}] {name}\n{"="*60}')
    cfg = TrainConfig(
        model_type=cfg_dict['model_type'],
        hidden=HIDDEN,
        modes=MODES,
        n_layers=N_LAYERS,
        data_dir=str(DATA_DIR),
        batch_size=BATCH_SIZE,
        n_epochs=N_EPOCHS,
        lr=2.5e-3,
        weight_decay=1e-4,
        warmup_steps=500,
        use_cosine_decay=cfg_dict['cosine'],
        input_noise_std=cfg_dict['noise'],
        device=device,
        seed=0,
        out_dir=str(RUNS_DIR / f"run_{i}"),
        max_train_seconds=15 * 60,
    )
    result = train(cfg)
    ablation_results[name] = result
    torch.save(ablation_results, RESULTS_DIR / 'ablation_results.pt')

In [ ]:
if not ablation_results:
    ablation_results = torch.load(RESULTS_DIR / 'ablation_results.pt', weights_only=False)

rows = []
for name, r in ablation_results.items():
    cfg = r['config']
    rows.append({
        'config': name,
        'model': cfg['model_type'],
        'cosine_LR': cfg['use_cosine_decay'],
        'noise_std': cfg['input_noise_std'],
        'test_NMSE': r['test_loss'],
        'best_val_NMSE': r['best_val_loss'],
        'params': r['n_params'],
    })
df_abl = pd.DataFrame(rows)
print(df_abl.to_string(index=False))
df_abl.to_csv(RESULTS_DIR / 'ablation_summary.csv', index=False)

In [ ]:
# Horizontal bar chart of N-MSE across configs
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['C0' if m == 'fno' else 'C1' for m in df_abl.model]
y_pos = np.arange(len(df_abl))
ax.barh(y_pos, df_abl.test_NMSE * 100, color=colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_abl.config)
ax.invert_yaxis()
ax.set_xlabel('Test N-MSE (%)')
ax.set_title(f'Training-trick ablation ({N_LAYERS}-layer models)')
for i, v in enumerate(df_abl.test_NMSE * 100):
    ax.text(v + 0.2, i, f'{v:.2f}%', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)

# Custom legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='C0', label='FNO'),
    Patch(color='C1', label='F-FNO')
], loc='lower right')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_bars.png', dpi=120)
plt.show()

## Interpretation

Expected patterns (from paper Fig. 3a):

- Adding cosine LR decay should help both models modestly
- Adding input noise should also help modestly — it acts as regularization
- The biggest gap should be between FNO and F-FNO even in matched trick
  conditions — showing that the architectural change (factorization +
  improved residuals) is doing real work, not just the training recipe

Caveat: training tricks have smaller effect at our short training budget.
Paper reports ~35% gain from FNO → F-FNO at 24 layers with 100k steps.
At our 8-layer, 15-epoch budget, the absolute gap will be smaller but the
direction should match.